## Phase 5 — Train ML Models

In [ ]:

from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
import time

# ── Define all 5 models with paper's exact parameters 
models = {
    'kNN':           KNeighborsClassifier(n_neighbors=8),
    'MLP':           MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42),
    'Random Forest': RandomForestClassifier(max_depth=16, n_estimators=50, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Grad. Boost':   GradientBoostingClassifier(n_estimators=100, random_state=42)
}

# ── Helper function: train + cross-validate one case 
def train_models(X_train, y_train, case_name):
    print(f"\n{'='*50}")
    print(f"  {case_name}")
    print(f"{'='*50}")
    trained = {}
    for name, model in models.items():
        start = time.time()
        # 5-fold cross-validation
        cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
        # Train on full training set
        model.fit(X_train, y_train)
        elapsed = time.time() - start
        trained[name] = model
        print(f"  {name:<20} CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}  ({elapsed:.1f}s)")
    return trained

#  Train for both cases 
print("Training started — this may take a few minutes...")

trained_bin = train_models(X_train_bin, y_train_bin, "Case 1 — Binary (Benign vs Darknet)")
trained_mul = train_models(X_train_mul, y_train_mul, "Case 2 — Multiclass (Tor/Non-Tor/VPN/Non-VPN)")

print("\nAll models trained successfully!")

In [ ]:
## Retrain MLP with Scaled Features
from sklearn.preprocessing import StandardScaler

#  Scale the features 
scaler = StandardScaler()

# Case 1
X_train_bin_scaled = scaler.fit_transform(X_train_bin)
X_test_bin_scaled  = scaler.transform(X_test_bin)

# Case 2 — fit a fresh scaler on multiclass train set
scaler2 = StandardScaler()
X_train_mul_scaled = scaler2.fit_transform(X_train_mul)
X_test_mul_scaled  = scaler2.transform(X_test_mul)

#Retrain MLP only
mlp_bin = MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42)
mlp_mul = MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42)

# Cross-validate
cv_bin = cross_val_score(mlp_bin, X_train_bin_scaled, y_train_bin, cv=cv, scoring='accuracy')
mlp_bin.fit(X_train_bin_scaled, y_train_bin)

cv_mul = cross_val_score(mlp_mul, X_train_mul_scaled, y_train_mul, cv=cv, scoring='accuracy')
mlp_mul.fit(X_train_mul_scaled, y_train_mul)

print("MLP (scaled) results:")
print(f"  Case 1 CV Accuracy: {cv_bin.mean():.4f} ± {cv_bin.std():.4f}")
print(f"  Case 2 CV Accuracy: {cv_mul.mean():.4f} ± {cv_mul.std():.4f}")

# ── Replace MLP in trained dicts with the scaled versions 
trained_bin['MLP'] = mlp_bin
trained_mul['MLP'] = mlp_mul

# Save scaled test sets for MLP evaluation in Phase 6
print("\nScaled MLP models saved. Ready for Phase 6!")